<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Persistent Storage Volumes

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook covers:** FABRIC offers **persistent storage volumes** -- dedicated storage that is tied to your project rather than to a specific slice. Unlike local disks and NVMe devices, persistent storage **survives slice deletion**, making it ideal for datasets, experiment results, and shared project data. This notebook shows how to attach, format, mount, and use a persistent storage volume.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Understand what **persistent storage volumes** are and how they differ from local disk and NVMe
2. Attach a persistent volume to a node using `node.add_storage()`
3. **Format** the volume (first-time use only)
4. **Mount** the volume and access stored data
5. Understand the volume lifecycle across multiple slices

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you should:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Successfully run the [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb) notebook
3. Have a **persistent volume already provisioned** for your project by your project lead

**Important:** Persistent volumes must be requested by your project lead from FABRIC administrators. You cannot create them yourself through the API. You will need to know the **volume name** and the **site** where it is located.

</div>

## Background: Persistent Storage vs. Other Storage Types

FABRIC offers three storage options, each with different persistence and performance characteristics:


**Key facts about persistent storage:**
- Tied to your **project**, not to any individual slice
- Located at a specific **site** -- your node must be at the same site
- Named following the convention: `<Project>_<site>_<size>_<index>` (e.g., `FABRIC_Staff_star_50G_1`)
- Must be **formatted once** (first use) and then just **mounted** on subsequent uses
- Data is preserved across slice deletions and new slice creations

<div class="fab-danger">

**Warning:** Formatting the volume (`mkfs.ext4`) erases all data on it. Only format the volume the **first time** you use it. On subsequent uses, skip the format step and go directly to mounting.

</div>

## What We're Building

In this notebook we will create a single node connected to a persistent storage volume.

<img src="./figs/slice_topology.png" width="40%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration
fablib.show_config();

---

## Step 2: Create a Slice with Persistent Storage

To attach a persistent volume, you must:
1. Place your node at the **same site** where the volume is located
2. Call `node.add_storage()` with the **exact volume name**

<div class="fab-warning">

**Tip:** Replace `site` and `storage_name` below with the actual values for your project's persistent volume. Ask your project lead if you do not know these values.

</div>

In [ ]:
# Replace with your project's volume name and site
site = 'STAR'
storage_name = f'FABRIC_Staff_star_50G_1'


# Create a new slice
slice = fablib.new_slice(name="MySlice")


# Add a node at the same site as the persistent volume
node = slice.add_node(name="Node1", site=site)

# Attach the persistent storage volume to this node
# The volume must already exist and be assigned to your project
node.add_storage(name=storage_name)


# Submit the slice and wait for provisioning
slice.submit();

---

## Step 3: Inspect the Slice

In [ ]:
# Display slice and node information
slice.show()
slice.list_nodes();

---

## Step 4: Format the Volume (First Time Only)

The first time you use a persistent volume, it is a raw block device with no filesystem. You need to format it with a filesystem (ext4 is recommended).

<div class="fab-danger">

**Warning:** Only run `mkfs.ext4` the **first time** you use this volume. Running it again on subsequent uses will **erase all previously stored data**. If your volume already contains data, skip this step and go directly to Step 5 (Mount).

</div>

In [ ]:
# Get the node and storage objects
node = slice.get_node('Node1')
storage = node.get_storage(storage_name)

# Get the device name (e.g., /dev/vdb) assigned to this volume
print(f"Storage Device Name: {storage.get_device_name()}")  

# Format the volume with ext4 filesystem
# WARNING: This erases all data on the volume!
stdout,stderr = node.execute(f"sudo mkfs.ext4 {storage.get_device_name()}")

---

## Step 5: Mount the Volume

After formatting (or on subsequent uses when the volume already has data), mount the volume to a directory on your node. Once mounted, you can read and write files just like any other directory.

<div class="fab-success">

**On subsequent uses:** Skip the `mkfs.ext4` step above and just run this mount step. Your previously stored data will be available at `/mnt/fabric_storage`.

</div>

In [ ]:
# Create a mount point directory and mount the volume
# Then verify with 'df -h' that the volume is mounted
stdout,stderr = node.execute(f"sudo mkdir /mnt/fabric_storage; "
                     f"sudo mount {storage.get_device_name()} /mnt/fabric_storage; "
                     f"df -h")

<div class="fab-success">

**What to look for:** In the `df -h` output, you should see the storage device (e.g., `/dev/vdb`) mounted at `/mnt/fabric_storage` with the expected size. You can now write files to this directory and they will persist even after you delete this slice.

</div>

---

## Step 6: Delete the Slice

<div class="fab-warning">

**Tip:** Deleting the slice releases the VM and the network resources, but your persistent storage volume and its data are **preserved**. The next time you create a slice at the same site and attach the same volume, your data will still be there.

</div>

In [ ]:
# Delete the slice (the persistent volume and its data are preserved)
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `storage not found` or slice fails | Volume name is incorrect or does not exist | Verify the exact volume name with your project lead |
| Slice fails with resource error | Node is not at the same site as the volume | Ensure `site` matches the volume's site |
| `mkfs.ext4` fails | Volume may already be mounted or in use | Unmount first with `sudo umount`, or check if another slice is using it |
| Data is missing after remounting | Volume was accidentally reformatted | Formatting erases all data -- only format once on first use |
| `Permission denied` writing to mount point | Mount point owned by root | Use `sudo chown` to change ownership, or write with `sudo` |
| `mkdir: cannot create directory`: already exists | Mount point from a previous use still exists | Skip the `mkdir` or add `-p` flag: `mkdir -p /mnt/fabric_storage` |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.show()` | Display slice attributes | [show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show) |
| `slice.list_nodes()` | List all nodes in the slice | [list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodes) |
| `slice.get_node(name)` | Get a specific node by name | [get_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_node) |
| `node.add_storage(name)` | Attach a persistent storage volume | [add_storage](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_storage) |
| `node.get_storage(name)` | Get a storage object by name | [get_storage](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.get_storage) |
| `storage.get_device_name()` | Get the block device path (e.g., /dev/vdb) | [get_device_name](https://fabric-fablib.readthedocs.io/en/latest/storage.html#fabrictestbed_extensions.fablib.storage.Storage.get_device_name) |
| `node.execute(command)` | Execute a shell command on the node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice (volume data is preserved) | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you understand persistent storage, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Local Disk** | [local_disk](../local_disk/local_disk.ipynb) | Customize built-in local disk sizes |
| **NVMe Storage** | [basic_nvme_devices](../basic_nvme_devices/basic_nvme_devices.ipynb) | Add fast NVMe storage devices (1 TB) to your nodes |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |
| **GPUs** | [fabric_gpu](../fabric_all_gpus/fabric_gpu.ipynb) | Use NVIDIA GPUs (T4, RTX6000, A30, A40) |